# Poster & comparison figures

Regenerates figures written to:
- `results/figures/Comparison of all models/` — unified leaderboard heatmaps, Wilcoxon summaries
- `results/figures/poster/` — held-out confusion matrices (best model per category)

Plotting logic lives in `unified_leaderboard.py`; this notebook calls the same functions in separate cells.

Model labels match `figures.ipynb` (`Random Forest`, `SVM`, `MERF (RF)`, etc.) via `DISPLAY_NAME` / `poster_display_name()`.

**Prerequisite:** Excel logs under `results/logs/**/*.xlsx` (GB, RF, SVM, MERF).

In [62]:
import importlib
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns

BASE = Path(r"C:/Users/janku/Documents/KCL/Research Project/Research Project")
if str(BASE) not in sys.path:
    sys.path.insert(0, str(BASE))

# Reload so edits to unified_leaderboard.py apply without restarting the kernel
import results.figures.unified_leaderboard as _unified_leaderboard
importlib.reload(_unified_leaderboard)

from results.figures.unified_leaderboard import (
    FIG_DIR,
    POSTER_DIR,
    POSTER_CLASSIFIER_ORDER,
    METRIC_DISPLAY_NAME,
    build_leaderboard,
    finalize_heatmap_figure,
    load_unified_tables,
    poster_display_name,
    rename_metric_columns,
)

sns.set_theme(style="whitegrid", context="talk")
FIG_DIR.mkdir(parents=True, exist_ok=True)
POSTER_DIR.mkdir(parents=True, exist_ok=True)

print(f"Comparison figures: {FIG_DIR}")
print(f"Poster figures:     {POSTER_DIR}")

Comparison figures: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\Comparison of all models
Poster figures:     C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\poster


In [63]:
METRICS_DIR = BASE / "results" / "metrics"

# Poster model names — same keys as figures.ipynb MODEL_FILES
MODEL_FILES = {
    ("RADAR", "Random Forest"): METRICS_DIR / "RF" / "RADAR" / "radar_rf_summary.csv",
    ("RADAR", "MERF (RF)"): METRICS_DIR / "RF" / "RADAR" / "radar_merf_summary.csv",
    ("RADAR", "SVM"): METRICS_DIR / "SVM" / "RADAR" / "radar_svm_cv_summary.csv",
    ("RADAR", "MERF (SVR)"): METRICS_DIR / "SVM" / "MERF" / "RADAR" / "radar_merf_svr_summary.csv",
    ("RADAR", "Gradient Boosting Classifier"): METRICS_DIR / "GradBoost" / "RADAR" / "radar_xgbc_cv_summary.csv",
    ("RADAR", "GPBoost"): METRICS_DIR / "GradBoost" / "RADAR" / "radar_gpboost_cv_summary.csv",
    ("ANDROIDS", "Random Forest"): METRICS_DIR / "RF" / "ANDROIDS" / "androids_rf_summary.csv",
    ("ANDROIDS", "MERF (RF)"): METRICS_DIR / "RF" / "ANDROIDS" / "androids_merf_summary.csv",
    ("ANDROIDS", "SVM"): METRICS_DIR / "SVM" / "ANDROIDS" / "androids_svm_cv_summary.csv",
    ("ANDROIDS", "MERF (SVR)"): METRICS_DIR / "SVM" / "MERF" / "Androids" / "androids_merf_summary.csv",
    ("ANDROIDS", "Gradient Boosting Classifier"): METRICS_DIR / "GradBoost" / "Androids" / "androids_gbc_cv_summary.csv",
    ("ANDROIDS", "GPBoost"): METRICS_DIR / "GradBoost" / "Androids" / "androids_gpboost_cv_summary.csv",
}

WILCOXON_HELDOUT_FILES = {
    ("RADAR", "Gradient Boosting Classifier"): METRICS_DIR / "radar_xgbc_wilcoxon_heldout.csv",
    ("RADAR", "GPBoost"): METRICS_DIR / "radar_gpboost_wilcoxon_heldout.csv",
    ("ANDROIDS", "Gradient Boosting Classifier"): METRICS_DIR / "GradBoost" / "Androids" / "androids_gbc_wilcoxon_heldout.csv",
    ("ANDROIDS", "GPBoost"): METRICS_DIR / "GradBoost" / "Androids" / "androids_gpboost_wilcoxon_heldout.csv",
}

tables = load_unified_tables()
cv_summary = tables["cv_summary"]
wilcoxon = tables["wilcoxon"]
confusion_matrices = tables["confusion_matrices"]

if cv_summary.empty:
    raise RuntimeError("No cv_summary rows in results/logs workbooks.")

leaderboard = build_leaderboard(cv_summary)
print(f"Models loaded: {len(leaderboard)}")
leaderboard[["dataset", "model", "display_name", "model_category", "roc_auc_mean", "mae_mean"]]

Models loaded: 18


,dataset,model,display_name,model_category,roc_auc_mean,mae_mean
0,Androids,Androids SVC,Support Vector Classifier,classifier,0.812722,NaN
1,Androids,Androids RFC,Random Forest Classifier,classifier,0.742203,NaN
2,Androids,Androids MERF-RF,ME RFRegressor,merf,0.697633,11.772584
3,Androids,Androids MERF-SVR,ME SVRegressor,merf,0.693425,11.662574
4,Androids,Androids SVR,Support Vector Regressor,regressor,0.687309,11.698885
5,Androids,Androids MERF-GBR,ME GBRegressor,merf,0.671804,11.814580
6,Androids,Androids GBC,Gradient Boosting Classifier,classifier,0.670635,NaN
7,Androids,Androids GPBoost,GPBoost,classifier,0.571429,NaN
8,RADAR,RADAR SVC,Support Vector Classifier,classifier,0.583951,NaN
9,RADAR,RADAR RFR,Random Forest Regressor,regressor,0.581766,4.980124


## Unified leaderboard heatmaps (tabular models)

Outputs:
- `unified_leaderboard_heatmap_classification_radar.png`
- `unified_leaderboard_heatmap_classification_androids.png`
- `unified_leaderboard_heatmap_regression_radar.png`
- `unified_leaderboard_heatmap_regression_androids.png`

In [64]:
import numpy as np
import pandas as pd

from results.figures.unified_leaderboard import (
    _heatmap_color_matrix,
    finalize_heatmap_figure,
    rename_metric_columns,
    save_fig,
)

heatmap_paths = []

# --- Classification heatmaps (per dataset) ---
cls_metrics = [c for c in METRIC_DISPLAY_NAME if c in leaderboard.columns and c.startswith(("accuracy", "f1", "roc_auc"))]
for dataset in ["RADAR", "Androids"]:
    sub = leaderboard[leaderboard["dataset"] == dataset].copy()
    if sub.empty:
        continue
    sub["poster_model"] = sub["display_name"].fillna(sub["model"].map(poster_display_name))
    cls_long = sub.melt(
        id_vars=["poster_model", "dataset"],
        value_vars=cls_metrics,
        var_name="metric",
        value_name="value",
    )
    cls_pivot = cls_long.pivot_table(
        index="poster_model", columns="metric", values="value", aggfunc="first"
    )
    cls_pivot = cls_pivot.dropna(how="all")
    if cls_pivot.empty:
        continue
    cls_pivot = rename_metric_columns(cls_pivot)

    fig_h = max(3.5, 0.55 * len(cls_pivot))
    fig, ax = plt.subplots(figsize=(8, fig_h))
    sns.heatmap(
        cls_pivot,
        annot=True,
        fmt=".3f",
        cmap="RdYlGn",
        vmin=0,
        vmax=1,
        ax=ax,
        cbar_kws={"label": "score"},
    )
    finalize_heatmap_figure(fig, ax, f"Classification metrics — {dataset} (CV means)")
    out = FIG_DIR / f"unified_leaderboard_heatmap_classification_{dataset.lower()}.png"
    heatmap_paths.append(save_fig(fig, out))

# --- Regression heatmaps (per dataset, column-normalized colours) ---
reg_metrics = [c for c in METRIC_DISPLAY_NAME if c in leaderboard.columns and c.startswith(("mae", "rmse", "r2"))]
higher_better = {"mae_mean": False, "rmse_mean": False, "r2_mean": True}
for dataset in ["RADAR", "Androids"]:
    reg_board = leaderboard[
        (leaderboard["dataset"] == dataset) & leaderboard[reg_metrics].notna().any(axis=1)
    ].copy()
    if reg_board.empty:
        continue
    reg_board["poster_model"] = reg_board["display_name"].fillna(
        reg_board["model"].map(poster_display_name)
    )
    reg_long = reg_board.melt(
        id_vars=["poster_model", "dataset"],
        value_vars=reg_metrics,
        var_name="metric",
        value_name="value",
    )
    reg_pivot = reg_long.pivot_table(
        index="poster_model", columns="metric", values="value", aggfunc="first"
    )
    reg_pivot = reg_pivot.dropna(how="all")
    if reg_pivot.empty:
        continue
    color_mat = _heatmap_color_matrix(reg_pivot, higher_better=higher_better)
    reg_display = rename_metric_columns(reg_pivot)
    color_display = rename_metric_columns(color_mat)
    fig, ax = plt.subplots(figsize=(8, max(5, 0.4 * len(reg_pivot))))
    sns.heatmap(
        color_display,
        annot=reg_display,
        fmt=".3f",
        cmap="RdYlGn",
        vmin=0,
        vmax=1,
        ax=ax,
        cbar_kws={"label": "normalized score"},
    )
    finalize_heatmap_figure(fig, ax, f"Regression metrics — {dataset} (CV means)")
    heatmap_paths.append(
        save_fig(fig, FIG_DIR / f"unified_leaderboard_heatmap_regression_{dataset.lower()}.png")
    )

for p in heatmap_paths:
    print(p)

Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\Comparison of all models\unified_leaderboard_heatmap_classification_radar.png
Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\Comparison of all models\unified_leaderboard_heatmap_classification_androids.png
Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\Comparison of all models\unified_leaderboard_heatmap_regression.png
C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\Comparison of all models\unified_leaderboard_heatmap_classification_radar.png
C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\Comparison of all models\unified_leaderboard_heatmap_classification_androids.png
C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\Comparison of all models\unified_leaderboard_heatmap_regression.png


## Transformer-based models (speech encoders)

Wav2Vec2, HuBERT, and Whisper linear probes on Androids, plus Kintsugi Health Whisper (HC/PT).

Metrics from `results/metrics/repr_learn/*_summary.csv` and `results/metrics/kintsugi_health/`.

Output: `unified_leaderboard_heatmap_speech_encoders.png`

In [65]:
from results.figures.unified_leaderboard import (
    REPR_LEARN_DIR,
    SPEECH_ENCODER_ORDER,
    TRANSFORMER_DISPLAY_NAMES,
    finalize_heatmap_figure,
    load_transformer_leaderboard,
    save_fig,
)

transformer_board = load_transformer_leaderboard()
if transformer_board.empty:
    print(f"No transformer metrics found (check {REPR_LEARN_DIR})")
else:
    show_cols = [
        c
        for c in ["dataset", "model", "accuracy_mean", "f1_mean", "roc_auc_mean", "eval_protocol"]
        if c in transformer_board.columns
    ]
    print(transformer_board[show_cols].to_string(index=False))

    cls_metrics = [
        c
        for c in METRIC_DISPLAY_NAME
        if c in transformer_board.columns and c.startswith(("accuracy", "f1", "roc_auc"))
    ]
    order = [
        TRANSFORMER_DISPLAY_NAMES[k]
        for k in SPEECH_ENCODER_ORDER
        if k in set(transformer_board["backbone"])
    ]

    tf_long = transformer_board.melt(
        id_vars=["model", "dataset", "backbone"],
        value_vars=cls_metrics,
        var_name="metric",
        value_name="value",
    )
    tf_pivot = tf_long.pivot_table(index="model", columns="metric", values="value", aggfunc="first")
    tf_pivot = tf_pivot.reindex(order).dropna(how="all")
    tf_pivot = rename_metric_columns(tf_pivot)

    fig, ax = plt.subplots(figsize=(8, max(3.8, 0.65 * len(tf_pivot))))
    sns.heatmap(
        tf_pivot,
        annot=True,
        fmt=".3f",
        cmap="RdYlGn",
        vmin=0,
        vmax=1,
        ax=ax,
        cbar_kws={"label": "score"},
    )
    ax.tick_params(axis="x", labelsize=10)
    finalize_heatmap_figure(
        fig, ax, "Classification metrics — speech encoders", ylabel="encoder / dataset"
    )

    speech_path = save_fig(fig, FIG_DIR / "unified_leaderboard_heatmap_speech_encoders.png")
    print(speech_path)

        dataset                   model  accuracy_mean  f1_mean  roc_auc_mean  eval_protocol
       Androids       HuBERT (Androids)       0.928283 0.928913      0.979700 5fold_group_cv
       Androids     Wav2Vec2 (Androids)       0.936970 0.939069      0.968876 5fold_group_cv
       Androids      Whisper (Androids)       0.830505 0.842202      0.940276 5fold_group_cv
Kintsugi Health Kintsugi Health (HC/PT)       0.846154 0.833333      0.964286  held_out_test
Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\Comparison of all models\unified_leaderboard_heatmap_speech_encoders.png
C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\Comparison of all models\unified_leaderboard_heatmap_speech_encoders.png


## Wilcoxon p-value summary plots

Held-out test Wilcoxon (best p per model vs baseline). Red dashed line = p = 0.05.

Outputs:
- `wilcoxon_pvalues_summary_radar.png`
- `wilcoxon_pvalues_summary_androids.png`

In [66]:
from results.figures.unified_leaderboard import _dataset_from_model, save_fig

plot_df = wilcoxon.copy()
plot_df["dataset"] = plot_df["model"].map(_dataset_from_model)
plot_df = (
    plot_df.sort_values("p_value")
    .groupby(["dataset", "model"], as_index=False)
    .first()
)
plot_df["neg_log10_p"] = -np.log10(plot_df["p_value"].clip(lower=1e-300))
plot_df["poster_model"] = plot_df["model"].map(poster_display_name)

wilcoxon_paths = []
for dataset in ["RADAR", "Androids"]:
    sub = plot_df[plot_df["dataset"] == dataset].sort_values("neg_log10_p", ascending=True)
    if sub.empty:
        continue
    fig, ax = plt.subplots(figsize=(10, max(4, 0.4 * len(sub))))
    color = "#4C72B0" if dataset == "RADAR" else "#DD8452"
    sns.barplot(data=sub, y="poster_model", x="neg_log10_p", color=color, ax=ax)
    ax.axvline(-np.log10(0.05), color="red", linestyle="--", linewidth=1, label="p = 0.05")
    ax.set_xlabel("-log10(p)")
    ax.set_title(f"Wilcoxon tests — {dataset} (best p per model)")
    ax.legend(loc="lower right")
    fig.tight_layout()
    out = FIG_DIR / f"wilcoxon_pvalues_summary_{dataset.lower()}.png"
    wilcoxon_paths.append(save_fig(fig, out))

for p in wilcoxon_paths:
    print(p)

plot_df[["dataset", "poster_model", "test_name", "p_value", "neg_log10_p"]]

Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\Comparison of all models\wilcoxon_pvalues_summary_radar.png
Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\Comparison of all models\wilcoxon_pvalues_summary_androids.png
C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\Comparison of all models\wilcoxon_pvalues_summary_radar.png
C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\Comparison of all models\wilcoxon_pvalues_summary_androids.png


,dataset,poster_model,test_name,p_value,neg_log10_p
0,Androids,Gradient Boosting Classifier,held_out_gbc_vs_stratified_dummy,2.122568e-01,0.673138
1,Androids,GPBoost,held_out_gbc_vs_gpboost,3.896273e-03,2.409351
2,Androids,ME GBRegressor,held_out_merf_gbr_vs_train_mean,6.035360e-01,0.219297
3,Androids,ME RFRegressor,held_out_merf_rf_vs_train_mean,6.725111e-01,0.172301
4,Androids,ME SVRegressor,held_out_merf_svr_vs_train_mean,5.482192e-02,1.261046
5,Androids,Random Forest Classifier,held_out_rfc_vs_stratified_dummy,9.367000e-02,1.028400
6,Androids,Support Vector Classifier,held_out_svc_vs_stratified_dummy,4.757570e-02,1.322615
7,Androids,Support Vector Regressor,held_out_svr_vs_train_mean,5.184248e-02,1.285314
8,RADAR,Gradient Boosting Classifier,held_out_gbc_vs_stratified_dummy,4.394725e-06,5.357068
9,RADAR,Gradient Boosting Regressor,held_out_gbr_vs_train_mean,1.616149e-01,0.791519


## Confusion matrices (held-out test)

Best model per dataset × category (classifier / regressor / MERF), then individual PNGs + grid.

Outputs under `results/figures/poster/`:
- `confusion_matrix_{dataset}_{category}_{model}.png` (one per panel)
- `confusion_matrices_best_per_category.png`

In [67]:
from results.figures.unified_leaderboard import (
    _confusion_matrix_array,
    _plot_confusion_matrix_ax,
    _save_confusion_figure,
    export_confusion_matrix,
    pick_best_models_per_category,
)

best_per_category = pick_best_models_per_category(leaderboard)
print(best_per_category[["dataset", "category", "display_name", "selection_metric", "selection_value"]].to_string(index=False))

# Individual held-out confusion matrices (best per category)
for _, row in best_per_category.iterrows():
    export_confusion_matrix(
        confusion_matrices,
        row["model"],
        category=row["category"],
        dataset=row["dataset"],
    )

# Grid figure
panels = []
for _, row in best_per_category.iterrows():
    cm = _confusion_matrix_array(confusion_matrices, row["model"], "held_out_test")
    if cm is not None:
        panels.append(row.to_dict() | {"cm": cm})

if panels:
    n = len(panels)
    ncols = min(3, n)
    nrows = int(np.ceil(n / ncols))
    with plt.rc_context({"axes.grid": False}):
        with sns.axes_style("white"):
            fig, axes = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 4 * nrows))
            axes = np.atleast_1d(axes).ravel()
            for ax, panel in zip(axes, panels):
                metric = panel["selection_metric"].replace("_mean", "")
                label = panel.get("display_name", poster_display_name(panel["model"]))
                title = (
                    f"{panel['dataset']} — {panel['category'].title()}\n"
                    f"{label}\n(best {metric}={panel['selection_value']:.3f})"
                )
                _plot_confusion_matrix_ax(ax, panel["cm"], title=title)
                ax.title.set_fontsize(10)
            for ax in axes[len(panels) :]:
                ax.axis("off")
            fig.suptitle("Held-out test confusion matrices — best model per category", y=0.98)
    grid_path = _save_confusion_figure(
        fig,
        POSTER_DIR / "confusion_matrices_best_per_category.png",
        subplots_adjust={
            "left": 0.06,
            "bottom": 0.06,
            "right": 0.98,
            "top": 0.90,
            "wspace": 0.45,
            "hspace": 0.55,
        },
    )
    print(grid_path)

 dataset   category              display_name selection_metric  selection_value
   RADAR classifier Support Vector Classifier     roc_auc_mean         0.583951
   RADAR  regressor  Support Vector Regressor         mae_mean         4.953497
   RADAR       merf            ME RFRegressor     roc_auc_mean         0.575329
Androids classifier Support Vector Classifier     roc_auc_mean         0.812722
Androids  regressor  Support Vector Regressor         mae_mean        11.698885
Androids       merf            ME RFRegressor     roc_auc_mean         0.697633
Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\poster\confusion_matrix_radar_classifier_radar_svc.png
Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\poster\confusion_matrix_radar_regressor_radar_svr.png
Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\poster\confusion_matrix_radar_merf_radar_merf-rf.png
Saved: C:\Users\jank

### Optional: single model confusion matrix

Uncomment and set `MODEL_NAME` to export one held-out matrix.

In [68]:
# from results.figures.unified_leaderboard import export_confusion_matrix
# MODEL_NAME = "Androids SVC"
# export_confusion_matrix(confusion_matrices, MODEL_NAME, matrix_name="held_out_test")